# Model Viewer App - One Click Install

Run the cell below to deploy the **Model Viewer** app to your workspace.  
It will create the app, deploy it, and print the shareable URL.

In [ ]:
APP_NAME = "model-viewer-app"
# GitHub fallback: where to fetch the app source if the workspace folder is missing.
# Change these two if you fork this installer for a different repo.
FALLBACK_REPO = "databricks-industry-solutions/lakehouse-business-data-models"
FALLBACK_BRANCH = "main"
FALLBACK_SUBPATH = "viewer/model-viewer-app"

import time, requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host, token = ctx.apiUrl().get(), ctx.apiToken().get()
H = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

nb = ctx.notebookPath().get()
ws_logical = "/".join(nb.split("/")[:-1]) + "/" + APP_NAME
src_fs = ws_logical if ws_logical.startswith("/Workspace/") else "/Workspace" + ws_logical

def _local_app_present():
    r = requests.get(f"{host}/api/2.0/workspace/get-status", headers=H,
                     params={"path": f"{ws_logical}/app.yaml"})
    return r.status_code == 200, r

def _install_from_github():
    """Fetch FALLBACK_SUBPATH from FALLBACK_REPO@FALLBACK_BRANCH and import every
    file into ws_logical. Raises Exception with a clear cause on any failure
    (network unreachable, repo/branch 404, no files matched, upload error)."""
    gh_h = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    tree_url = (f"https://api.github.com/repos/{FALLBACK_REPO}/git/trees/"
                f"{FALLBACK_BRANCH}?recursive=1")
    try:
        tr = requests.get(tree_url, headers=gh_h, timeout=15)
    except requests.exceptions.RequestException as ne:
        raise Exception(f"network unreachable while fetching {tree_url}: {ne}")
    if tr.status_code == 404:
        raise Exception(f"repo or branch not found: {FALLBACK_REPO}@{FALLBACK_BRANCH} "
                        f"(GitHub returned 404 for {tree_url})")
    if tr.status_code in (401, 403):
        raise Exception(f"GitHub denied access to {FALLBACK_REPO}@{FALLBACK_BRANCH} "
                        f"({tr.status_code}); is the repo private or rate-limited? "
                        f"Body: {tr.text[:200]}")
    if tr.status_code != 200:
        raise Exception(f"unexpected GitHub response {tr.status_code} for {tree_url}: "
                        f"{tr.text[:200]}")

    prefix = FALLBACK_SUBPATH.rstrip("/") + "/"
    files = [e for e in (tr.json().get("tree") or [])
             if e.get("type") == "blob" and (e.get("path") or "").startswith(prefix)]
    if not files:
        raise Exception(f"no files found under '{FALLBACK_SUBPATH}/' in "
                        f"{FALLBACK_REPO}@{FALLBACK_BRANCH} (tree returned {len(tr.json().get('tree') or [])} entries total)")

    print(f"       Fetching {len(files)} file(s) from {FALLBACK_REPO}@{FALLBACK_BRANCH}/{FALLBACK_SUBPATH}/ ...")
    mr = requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=H, json={"path": ws_logical})
    if mr.status_code not in (200, 201):
        raise Exception(f"workspace mkdirs failed for {ws_logical}: {mr.status_code} {mr.text[:200]}")

    for entry in files:
        rel = entry["path"][len(prefix):]
        if not rel:
            continue
        try:
            br = requests.get(
                f"https://api.github.com/repos/{FALLBACK_REPO}/git/blobs/{entry['sha']}",
                headers=gh_h, timeout=30,
            )
        except requests.exceptions.RequestException as ne:
            raise Exception(f"network unreachable while fetching blob {rel}: {ne}")
        if br.status_code != 200:
            raise Exception(f"blob fetch failed for {rel} ({br.status_code}): {br.text[:200]}")
        content_b64 = (br.json().get("content") or "").replace("\n", "")
        if not content_b64:
            raise Exception(f"blob {rel} returned empty content (size={entry.get('size')})")

        target = f"{ws_logical}/{rel}"
        parent = target.rsplit("/", 1)[0]
        if parent != ws_logical:
            requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=H, json={"path": parent})
        ir = requests.post(
            f"{host}/api/2.0/workspace/import", headers=H,
            json={"path": target, "format": "AUTO", "content": content_b64, "overwrite": True},
        )
        if ir.status_code not in (200, 201):
            raise Exception(f"workspace import failed for {target}: {ir.status_code} {ir.text[:200]}")
        print(f"         + {rel}")

    ok, _ = _local_app_present()
    if not ok:
        raise Exception(f"files uploaded but app.yaml still not visible at {ws_logical}/app.yaml")

local_ok, local_resp = _local_app_present()
if local_ok:
    print(f"[1/4] Source verified locally: {ws_logical}")
else:
    print(f"[1/4] Local source not found at {ws_logical}/app.yaml "
          f"(workspace get-status: {local_resp.status_code}); "
          f"falling back to GitHub: {FALLBACK_REPO}@{FALLBACK_BRANCH}/{FALLBACK_SUBPATH}/")
    try:
        _install_from_github()
    except Exception as fe:
        raise Exception(
            f"App source not available.\n"
            f"  - Local check FAILED: {ws_logical}/app.yaml not found in workspace.\n"
            f"  - GitHub fallback FAILED: {fe}\n"
            f"Fix one of the following:\n"
            f"  (a) Sync the 'viewer/' folder from the repo into your workspace alongside this notebook, OR\n"
            f"  (b) Ensure this cluster can reach api.github.com and the repo "
            f"'{FALLBACK_REPO}@{FALLBACK_BRANCH}' is accessible.\n"
        )
    print(f"       ✓ App source materialized at {ws_logical}")
print(f"       Apps API source_code_path: {src_fs}")

def _wait_compute_active(timeout_s=600):
    deadline = time.time() + timeout_s
    last = ""
    while time.time() < deadline:
        info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
        cs = info.get("compute_status", {}).get("state", "")
        if cs != last:
            print(f"       Compute: {cs}")
            last = cs
        if cs == "ACTIVE":
            return info
        if cs in ("ERROR", "STOPPED"):
            raise Exception(f"Compute entered {cs} while waiting; aborting deploy.")
        time.sleep(10)
    raise Exception(f"Compute did not reach ACTIVE within {timeout_s}s (last state={last!r}).")

r = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H)
if r.status_code == 200:
    print(f"[2/4] App '{APP_NAME}' exists, ensuring compute is running...")
    cs = r.json().get("compute_status", {}).get("state", "")
    if cs != "ACTIVE":
        if cs in ("STOPPED", "ERROR", ""):
            sr = requests.post(f"{host}/api/2.0/apps/{APP_NAME}/start", headers=H, json={})
            assert sr.status_code in (200, 201, 202), f"Failed to start app compute: {sr.status_code} {sr.text}"
        _wait_compute_active()
elif r.status_code == 404:
    print(f"[2/4] Creating app '{APP_NAME}'...")
    cr = requests.post(f"{host}/api/2.0/apps", headers=H, json={"name": APP_NAME, "description": "Data Model Viewer - interactive graph visualization"})
    assert cr.status_code in (200, 201), f"Failed to create app: {cr.status_code} {cr.text}"
    print(f"       App created. Waiting for compute to become ACTIVE...")
    _wait_compute_active()
else:
    raise Exception(f"Unexpected response checking app: {r.status_code} {r.text}")

print(f"[3/4] Deploying...")
r = requests.post(
    f"{host}/api/2.0/apps/{APP_NAME}/deployments",
    headers=H,
    json={"source_code_path": src_fs, "mode": "SNAPSHOT"},
)
assert r.status_code in (200, 201), f"Deploy failed: {r.status_code} {r.text}"
deployment_id = r.json().get("deployment_id", "")
print(f"       Deployment started: {deployment_id}")

last_state = ""
for _ in range(60):
    time.sleep(10)
    info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
    dep = info.get("active_deployment") or info.get("pending_deployment") or {}
    status = dep.get("status", {})
    state = status.get("state", "")
    msg = status.get("message", "")
    if state != last_state:
        print(f"       {state}: {msg}")
        last_state = state
    if state == "SUCCEEDED":
        url = info.get("url", "")
        app_state = info.get("app_status", {}).get("state", "")
        print(f"[4/4] Deployed (app_status={app_state}).")
        print(f"\n{'='*60}\n  App URL: {url}\n{'='*60}")
        try:
            displayHTML(
                f'<h2 style="color:#4ECDC4">Model Viewer Deployed!</h2>'
                f'<p style="font-size:16px"><a href="{url}" target="_blank">{url}</a></p>'
                f'<p>Upload a <code>model.json</code> to visualize your data model.</p>'
            )
        except Exception:
            pass
        break
    if state == "FAILED":
        raise Exception(f"Deployment failed: {msg}")
else:
    raise Exception("Deployment did not complete within 10 minutes. Check app status in the workspace UI.")